In [ ]:
!apt-get install -y tesseract-ocr
!pip install pytesseract opencv-python-headless scikit-learn
!pip install pandas numpy matplotlib pillow pypdf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (5.3.4-1build5).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [ ]:
import os
import cv2
import re
import numpy as np
import pandas as pd
import pytesseract
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from google.colab import files
from pypdf import PdfWriter

In [ ]:
uploaded = files.upload()

image_paths = []

for filename in uploaded.keys():

    if filename.lower().endswith(
        ('.jpg', '.jpeg', '.png', '.tif', '.tiff')
    ):
        image_paths.append(filename)

print("Total images uploaded:", len(image_paths))

Total images uploaded: 0


In [ ]:
def preprocess_image(image_path):

    image = cv2.imread(image_path)

    gray = cv2.cvtColor(
        image, cv2.COLOR_BGR2GRAY
    )

    gray = cv2.GaussianBlur(
        gray, (3, 3), 0
    )

    processed = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )[1]

    return processed


def extract_text(image_path):

    processed = preprocess_image(image_path)

    text = pytesseract.image_to_string(
        processed,
        config="--psm 6"
    )

    return text

In [ ]:
training_data = {

    "Invoice": [
        "invoice invoice number billing amount total tax GST",
        "bill payment quantity unit price customer supplier",
        "purchase invoice subtotal grand total due date",
        "tax invoice product description payment terms"
    ],

    "Application": [
        "application form applicant personal information",
        "name address date of birth application number",
        "declaration signature applicant details",
        "personal details contact information form"
    ],

    "Bank Statement": [
        "bank statement account number balance",
        "transaction date debit credit account holder",
        "opening balance closing balance bank transaction",
        "withdrawal deposit statement period"
    ],

    "Identity Document": [
        "identity card identification date of birth",
        "passport passport number nationality photograph",
        "government identity proof address card",
        "identity number holder name date of birth"
    ]
}

texts = []
labels = []

for category, examples in training_data.items():

    for example in examples:

        texts.append(example)
        labels.append(category)

print("Training samples:", len(texts))

Training samples: 16


In [ ]:
classifier = Pipeline([

    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2)
    )),

    ("model", MultinomialNB(alpha=0.5))

])

classifier.fit(texts, labels)

print("Probabilistic classifier trained successfully!")

Probabilistic classifier trained successfully!


In [ ]:
def classify_page(text):

    probabilities = classifier.predict_proba(
        [text]
    )[0]

    classes = classifier.classes_

    best_index = np.argmax(probabilities)

    predicted_class = classes[best_index]

    confidence = probabilities[best_index]

    probability_dict = dict(
        zip(classes, probabilities)
    )

    return predicted_class, confidence, probability_dict

In [ ]:
class DocumentFSM:

    def __init__(self):

        self.state = "WAITING"

        self.current_document = []

        self.documents = []

        self.current_type = None

    def process_page(self, page_info):

        page_class = page_info["class"]

        confidence = page_info["confidence"]

        # Start the first document
        if self.state == "WAITING":

            self.current_document = [page_info]

            self.current_type = page_class

            self.state = "ACTIVE"

        elif self.state == "ACTIVE":

            # Check whether a new document starts
            new_document = (
                page_class != self.current_type
                and confidence >= 0.50
            )

            if new_document:

                # Complete previous document
                self.documents.append(
                    self.current_document
                )

                self.state = "COMPLETED"

                # Start next document
                self.current_document = [page_info]

                self.current_type = page_class

                self.state = "ACTIVE"

            else:

                self.current_document.append(page_info)

    def finish(self):

        if self.current_document:

            self.documents.append(
                self.current_document
            )

        self.state = "COMPLETED"

        return self.documents

In [ ]:
fsm = DocumentFSM()

page_results = []

for i, image_path in enumerate(image_paths):

    print(f"Processing page {i+1}: {image_path}")

    # Extract OCR text
    text = extract_text(image_path)

    # Predict document category
    predicted_class, confidence, probs = classify_page(text)

    page_info = {

        "page": i + 1,

        "filename": image_path,

        "text": text,

        "class": predicted_class,

        "confidence": confidence,

        "probabilities": probs
    }

    page_results.append(page_info)

    # Pass page to FSM
    fsm.process_page(page_info)


documents = fsm.finish()

print("\nDocument processing completed!")
print("Total documents detected:", len(documents))


Document processing completed!
Total documents detected: 0


In [ ]:
results = []

for page in page_results:

    results.append({

        "Page": page["page"],

        "Filename": page["filename"],

        "Predicted Document": page["class"],

        "Confidence": round(
            page["confidence"] * 100, 2
        )

    })

df = pd.DataFrame(results)

display(df)

""


In [ ]:
os.makedirs("Separated_Documents", exist_ok=True)

for i, document in enumerate(documents):

    pdf_images = []

    for page in document:

        image = Image.open(
            page["filename"]
        ).convert("RGB")

        pdf_images.append(image)

    if pdf_images:

        output_file = (
            f"Separated_Documents/Document_{i+1}.pdf"
        )

        pdf_images[0].save(
            output_file,
            save_all=True,
            append_images=pdf_images[1:]
        )

        print("Created:", output_file)

In [ ]:
print("\n========== FINAL RESULTS ==========")

for i, document in enumerate(documents):

    print(f"\nDocument {i+1}")

    print("Document Type:", document[0]["class"])

    print("Total Pages:", len(document))

    print(
        "Pages:",
        [page["page"] for page in document]
    )

    print(
        "Average Confidence:",
        round(
            np.mean([
                page["confidence"]
                for page in document
            ]) * 100, 2
        ),
        "%"
    )


========== FINAL RESULTS ==========


In [ ]:
import shutil

shutil.make_archive(
    "Separated_Documents",
    "zip",
    "Separated_Documents"
)

files.download("Separated_Documents.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>